In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

load_dotenv()
llm = ChatOpenAI(
            model_name = os.getenv("GITHUB_MODEL", "openai/gpt-4o"),
            openai_api_base="https://models.github.ai/inference",
            openai_api_key=os.environ["GITHUB_TOKEN"])

In [19]:
speech="""
People across the country, involved in government, political, and social activities, are dedicating their time to make the ‘Viksit Bharat Sankalp Yatra’ (Developed India Resolution Journey) successful. Therefore, as a Member of Parliament, it was my responsibility to also contribute my time to this program. So, today, I have come here just as a Member of Parliament and your ‘sevak’, ready to participate in this program, much like you.

In our country, governments have come and gone, numerous schemes have been formulated, discussions have taken place, and big promises have been made. However, my experience and observations led me to believe that the most critical aspect that requires attention is ensuring that the government’s plans reach the intended beneficiaries without any hassles. If there is a ‘Pradhan Mantri Awas Yojana’ (Prime Minister’s housing scheme), then those who are living in jhuggis and slums should get their houses. And he should not need to make rounds of the government offices for this purpose. The government should reach him. Since you have assigned this responsibility to me, about four crore families have got their ‘pucca’ houses. However, I have encountered cases where someone is left out of the government benefits. Therefore, I have decided to tour the country again, to listen to people’s experiences with government schemes, to understand whether they received the intended benefits, and to ensure that the programs are reaching everyone as planned without paying any bribes. We will get the real picture if we visit them again. Therefore, this ‘Viksit Bharat Sankalp Yatra’ is, in a way, my own examination. I want to hear from you and the people across the country whether what I envisioned and the work I have been doing aligns with reality and whether it has reached those for whom it was meant.

It is crucial to check whether the work that was supposed to happen has indeed taken place. I recently met some individuals who utilized the Ayushman card to get treatment for serious illnesses. One person met with a severe accident, and after using the card, he could afford the necessary operation, and now he is recovering well. When I asked him, he said: “How could I afford this treatment? Now that there is the Ayushman card, I mustered courage and underwent an operation. Now I am perfectly fine.”  Such stories are blessings to me.

The bureaucrats, who prepare good schemes, expedite the paperwork and even allocate funds, also feel satisfied that 50 or 100 people who were supposed to get the funds have got it. The funds meant for a thousand villages have been released. But their job satisfaction peaks when they hear that their work has directly impacted someone’s life positively. When they see the tangible results of their efforts, their enthusiasm multiplies. They feel satisfied. Therefore, ‘Viksit Bharat Sankalp Yatra’ has had a positive impact on government officers. It has made them more enthusiastic about their work, especially when they witness the tangible benefits reaching the people. Officers now feel satisfied with their work, saying, “I made a good plan, I created a file, and the intended beneficiaries received the benefits.” When they find that the money has reached a poor widow under the Jeevan Jyoti scheme and it was a great help to her during her crisis, they realise that they have done a good job. When a government officer listens to such stories, he feels very satisfied.

There are very few who understand the power and impact of the ‘Viksit Bharat Sankalp Yatra’. When I hear people connected to bureaucratic circles talking about it, expressing their satisfaction, it resonates with me. I’ve heard stories where someone suddenly received 2 lakh rupees after the death of her husband, and a sister mentioned how the arrival of gas in her home transformed her lives. The most significant aspect is when someone says that the line between rich and poor has vanished. While the slogan ‘Garibi Hatao’ (Remove Poverty) is one thing, but the real change happens when a person says, “As soon as the gas stove came to my house, the distinction between poverty and affluence disappeared.
"""

In [20]:
# Set up the messages for the chat model
chat_message=[
    SystemMessage(content="You are a helpful assistant that summarizes text."),
    HumanMessage(content=f"Summarize the following speech in 100 words:Text{speech}"),
]

In [21]:

tokens_in_speech = llm.get_num_tokens(chat_message[-1].content)
print(f"Number of tokens in the speech: {tokens_in_speech}")

Number of tokens in the speech: 863


1st way of Summarization using the chat_message

In [22]:
## 1st way of Summarization using the chat_message
llm_response = llm.invoke(chat_message)
print("Summary of the speech:")
print(llm_response.content)

Summary of the speech:
In a speech about the 'Viksit Bharat Sankalp Yatra' (Developed India Resolution Journey), a Member of Parliament expressed the importance of ensuring government benefits reach their intended recipients. Emphasizing his role as a 'sevak' (servant) of the people, he highlighted the need for programs like the Pradhan Mantri Awas Yojana to effectively support those in need without bureaucratic hurdles. The MP plans to tour the country to gather feedback on government schemes and assess their impact. He shared inspiring stories of individuals benefiting from programs such as the Ayushman card, noting that witnessing tangible results motivates both beneficiaries and bureaucrats alike.


2nd way of Summarization using the simple prompt template

In [12]:
## 2nd way of Summarization using the simple prompt template
from langchain_core.prompts import PromptTemplate #--> used for text or string inputs

generictemplate = """
Summarize the following speech in 100 words:
Text: {speech}
Covert this to {language}
"""


prompt = PromptTemplate(
    input_variables=["speech", "language"],
    template=generictemplate
)
prompt_format = prompt.format(speech=speech, language="English")


In [17]:
llm.get_num_tokens(prompt_format)

chain = prompt | llm
summary = chain.invoke({"speech": speech, "language": "Hindi"})
print("Summary of the speech using PromptTemplate:")
print(summary)

Summary of the speech using PromptTemplate:
content="देशभर में सरकारी, राजनीतिक और सामाजिक गतिविधियों से जुड़े लोग 'विकसित भारत संकल्प यात्रा' को सफल बनाने में अपना समय समर्पित कर रहे हैं। इसलिए, एक सांसद के रूप में, मुझे भी इस कार्यक्रम में योगदान देना आवश्यक था। मेरे अनुभव से, सबसे महत्वपूर्ण बात यह है कि सरकारी योजनाएं लाभार्थियों तक बिना किसी बाधा के पहुंचें। मैंने देखा है कि कई लोग सरकारी लाभों से वंचित रह जाते हैं। मैं लोगों के अनुभव सुनने और यह सुनिश्चित करने के लिए देशभर में यात्रा करने का निर्णय लिया है कि योजनाएं सही तरीके से लागू हो रही हैं। ‘विकसित भारत संकल्प यात्रा’ ने सरकारी अधिकारियों में कार्य के प्रति उत्साह बढ़ाया है, क्योंकि जब वे देख पाते हैं कि उनके कार्यों का सकारात्मक प्रभाव पड़ रहा है, तो उन्हें संतोष मिलता है। लोगों की कहानियाँ सुनकर अधिकारियों को एहसास होता है कि उन्होंने अच्छा काम किया है। सबसे महत्वपूर्ण परिवर्तन तब होता है जब लोगों की ज़िंदगी में सुधार दिखाई देता है, जैसे कि गैस आने से गरीब और अमीर के बीच की रेखा मिट गई हो।" additional_kwargs={'refusal': N

3rd way of summarization using StuffDocumentChain

In [5]:
from langchain_community.document_loaders import PyPDFLoader

In [6]:
loader = PyPDFLoader("apjspeech.pdf")
docs = loader.load_and_split()
docs

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealt

In [14]:
## Summarize using StuffDocumentChain
from langchain_core.prompts import PromptTemplate #--> used for text or string inputs
template="""Write a concise summary of the following speech:
Text: {text}"""

stuff_prompt = PromptTemplate(
    input_variables=["text"],
    template=template
)

In [17]:
stuff_chain = stuff_prompt | llm
combined_text = "\n\n".join([doc.page_content for doc in docs])

summary = stuff_chain.invoke({"text": combined_text})
print("Summary of the speech using StuffDocumentChain:")
print(summary)

Summary of the speech using StuffDocumentChain:
content='In his departing speech, Dr. A. P. J. Abdul Kalam expressed gratitude for his five years in Rashtrapati Bhavan and shared key insights gained from interactions with diverse groups across India. He emphasized the importance of youth aspirations, village empowerment, and agricultural growth, highlighting the need for unity and development. Kalam underscored the value of overcoming challenges through partnerships and courage, sharing inspiring stories from those facing adversity. He proposed initiatives for societal transformation through connectivity and technology, and lauded the crucial role of the defense forces in safeguarding the nation. Dr. Kalam concluded with a vision of a developed India by 2020, calling for collective effort to bridge rural-urban divides, ensure equitable resource distribution, and advance education and healthcare. He thanked the citizens for their support and reaffirmed his commitment to the mission of m

3rd way of summarization using MapReduce - if it does not fit the LLM context video
Use tokens = llm.get_num_tokens(text) to find out the token size that teh context supports

In [7]:
##Step 1 — Map chain
from langchain_core.prompts import PromptTemplate #--> used for text or string inputs
from langchain_core.output_parsers import StrOutputParser #--> used to parse string outputs since messages are AIMessage objects
from langchain_text_splitters import RecursiveCharacterTextSplitter #--> used to split text into smaller chunks

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
final_documents = text_splitter.split_documents(docs)
final_documents



[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology,'),
 Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'aut

In [34]:
##Step 1 — Map chain
map_prompt = PromptTemplate(
    input_variables=["text"],
    template="Summarize the following text:\n\n{text}\n\nSummary:"
)

In [35]:
map_chain = map_prompt | llm | StrOutputParser()

In [ ]:
##Step 2 — Reduce chain
reduce_prompt = PromptTemplate(
    input_variables=["text"],
    template="Combine the following summaries into a final concise summary:\n\n{text}\n\nSummary:"
)

reduce_chain = reduce_prompt | llm | StrOutputParser()

In [ ]:
## Step 3 — Map-Reduce Document Chain

# MAP
inputs = [{"text": doc.page_content} for doc in final_documents]
map_results = map_chain.batch(inputs) #--> batch supports parallel execution instead of sequential call
summaries = [res.content for res in map_results]

# REDUCE
final_summary = reduce_chain.invoke({"text": "\n\n".join(summaries)})

In [28]:
final_summary

'In his farewell address, Dr. A. P. J. Abdul Kalam reflects on his rewarding tenure at Rashtrapati Bhavan and expresses gratitude for the diverse associations encountered. He outlines ten key messages to drive India\'s development, emphasizing the empowerment of youth, villages, and rural competencies, as well as fostering agricultural growth and national pride. Notably, he recalls a poignant interaction with a schoolgirl who questioned India’s development status by 2020, underscoring the crucial role of youth aspirations.\n\nKalam highlights transformative initiatives, such as the Providing Urban Amenities in Rural Areas (PURA) project, which has positively impacted villages, enhancing connectivity and job creation. He shares experiences with farmers, emphasizing agricultural productivity, and lauds the resilience of communities affected by disasters, such as the 2005 earthquake and the 2004 tsunami.\n\nHe also discusses the Pan African e-Network project aimed at enhancing communicati

4th way - Refine Chain Summarization

In [9]:
from langchain_core.prompts import ChatPromptTemplate

initial_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that summarizes text."),
    ("human", "Write a concise summary of the following:\n\n{text}")
])

refine_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant refining an existing summary."),
    ("human",
     "Existing summary:\n{existing_summary}\n\n"
     "New context:\n{text}\n\n"
     "Refine the summary to include the new information."
    )
])


In [10]:
summary = None

for doc in final_documents:
    if summary is None:
        summary = llm.invoke(
            initial_prompt.format_messages(text=doc.page_content)
        ).content
    else:
        summary = llm.invoke(
            refine_prompt.format_messages(
                existing_summary=summary,
                text=doc.page_content
            )
        ).content

RateLimitError: Too many requests. For more on scraping GitHub and how it may affect your rights, please review our Terms of Service (https://docs.github.com/en/site-policy/github-terms/github-terms-of-service).

In [ ]:
print(summary)